# Baseline: Markov Chain Music Model

This notebook implements a 2nd-order Markov Chain music model as a baseline.

In [1]:
import os, sys, json, glob, random
import numpy as np
import pretty_midi
import matplotlib.pyplot as plt
from collections import defaultdict, Counter

sys.path.insert(0, '../src')
from evaluation.pitch_histogram import compute_pitch_histogram, histogram_similarity
from evaluation.rhythm_score import rhythm_diversity, repetition_ratio

In [ ]:
class MarkovMusicModel:
    '''2nd-order Markov Chain model for music generation.
    State = (pitch_t-1, pitch_t); transition -> pitch_t+1
    '''
    def __init__(self, order=2):
        self.order = order
        self.transitions = defaultdict(Counter)
        self.durations   = defaultdict(list)

    def train(self, sequences):
        for seq in sequences:
            pitches = [e['pitch'] for e in seq]
            durs    = [e['duration'] for e in seq]
            for i in range(len(pitches) - self.order):
                state = tuple(pitches[i:i+self.order])
                nxt   = pitches[i+self.order]
                self.transitions[state][nxt] += 1
                self.durations[state].append(durs[i+self.order])

    def generate(self, length=64, seed=None):
        if seed is None:
            seed = random.choice(list(self.transitions.keys()))
        sequence = list(seed)
        for _ in range(length - self.order):
            state = tuple(sequence[-self.order:])
            if state not in self.transitions:
                state = random.choice(list(self.transitions.keys()))
            counts  = self.transitions[state]
            total   = sum(counts.values())
            choices = list(counts.keys())
            probs   = [counts[c]/total for c in choices]
            nxt = random.choices(choices, weights=probs)[0]
            sequence.append(nxt)
        return sequence

In [ ]:
# Load processed events
PROCESSED_DIR = '../data/processed'
json_files = glob.glob(os.path.join(PROCESSED_DIR, '*.json'))
sequences = []
for jf in json_files[:500]:
    with open(jf) as f:
        d = json.load(f)
    sequences.append(d['events'])
print(f'Loaded {len(sequences)} sequences')

In [ ]:
model = MarkovMusicModel(order=2)
model.train(sequences)
print(f'Unique states: {len(model.transitions)}')

In [ ]:
# Generate and evaluate
generated = [model.generate(length=128) for _ in range(20)]

diversity_scores = [rhythm_diversity([0.25]*len(g)) for g in generated]
repetition_scores = [repetition_ratio(g) for g in generated]

print(f'Mean Rhythm Diversity : {np.mean(diversity_scores):.3f}')
print(f'Mean Repetition Ratio : {np.mean(repetition_scores):.3f}')

In [ ]:
# Plot pitch histogram of generated vs training
gen_pitches  = [p for g in generated for p in g]
real_pitches = [e['pitch'] for s in sequences[:100] for e in s]

h_gen  = compute_pitch_histogram(gen_pitches)
h_real = compute_pitch_histogram(real_pitches)

fig, axes = plt.subplots(1, 2, figsize=(12,4))
axes[0].bar(range(12), h_real, label='Real'); axes[0].set_title('Real Pitch Distribution')
axes[1].bar(range(12), h_gen,  label='Markov', color='orange'); axes[1].set_title('Markov Generated')
plt.tight_layout(); plt.savefig('../outputs/plots/markov_pitch_hist.png', dpi=150); plt.show()
print(f'L1 Similarity: {histogram_similarity(h_gen, h_real):.4f}')